# Notebook 12 — ordinal do colega x base limpa (compounding)

Receita do colega (registo Sec.19, sub6/sub7, CV 0.506 / público 0.51): Frank & Hall com 15 binários XGBoost (300 árvores, depth 4, lr 0.05, min_child 3, subsample 0.8, colsample 0.5, sample_weight balanced), P(classe) por diferenças com monotonização, correção `max(pred, FAIXA_MINIMA)`.
E23: F&H 1 semente na base 311 (isola efeito da base; ref colega 0.5008 na 339). E24: igual + quarentena-10 no train (isola efeito limpeza). E25 (5 sementes + reg) só se E24 prometer. Mesmas folds seed 42: OOFs comparáveis.

In [1]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
K = 16
TARGET = 'AGE_RANGE'

def _data_dir():
    for cand in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError('pasta data/ nao encontrada')
DATA_DIR = _data_dir()

train = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = train[TARGET].to_numpy().astype(int)
Xv = train.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
fmin = train['FAIXA_MINIMA'].to_numpy().astype(int)
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(Xv, y))
print('X:', Xv.shape)

def xgb_bin(seed=RANDOM_STATE):
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
        min_child_weight=3, subsample=0.8, colsample_bytree=0.5,
        tree_method='hist', random_state=seed, n_jobs=-1, eval_metric='logloss')

def fh_oof(drop_idx=None, tag='e23', seed=RANDOM_STATE):
    p = os.path.join(DATA_DIR, f'nb12_{tag}.npz')
    if os.path.exists(p):
        d = np.load(p)
        print(f'{tag}: cache {p}')
        return d['f1s'], d['oof'], d['proba']
    t0 = time.time()
    f1s, oof, proba = [], np.zeros(len(y), dtype=int), np.zeros((len(y), K))
    drop = set() if drop_idx is None else set(np.asarray(drop_idx).tolist())
    for fi, (tr, va) in enumerate(FOLDS):
        atr = np.array([i for i in tr if i not in drop])
        sw = compute_sample_weight('balanced', y[atr])
        pg = np.zeros((len(va), K - 1))
        for k in range(K - 1):
            m = xgb_bin(seed)
            m.fit(Xv[atr], (y[atr] > k).astype(int), sample_weight=sw)
            pg[:, k] = m.predict_proba(Xv[va])[:, 1]
        pg = np.minimum.accumulate(pg, axis=1)
        pfull = np.zeros((len(va), K))
        prev = np.ones(len(va))
        for k in range(K):
            cur = pg[:, k] if k < K - 1 else np.zeros(len(va))
            pfull[:, k] = np.clip(prev - cur, 0, None)
            prev = cur
        pr = pfull.argmax(axis=1)
        oof[va] = pr
        proba[va] = pfull
        f1s.append(f1_score(y[va], pr, average='macro'))
        print(f'  fold {fi}: {f1s[-1]:.4f} ({time.time()-t0:.0f}s)', flush=True)
    f1s = np.array(f1s)
    np.savez_compressed(p, f1s=f1s, oof=oof, proba=proba)
    return f1s, oof, proba

def corrigir(pred):
    return np.maximum(pred, fmin)

RESULTADOS = []
def registar(exp, variante, f1s, ref, nota=''):
    f1s = np.asarray(f1s, float)
    r = {'exp': exp, 'variante': variante, 'f1': float(f1s.mean()), 'std': float(f1s.std()),
         'delta': float(f1s.mean() - ref), 'nota': nota}
    RESULTADOS.append(r)
    print(r)
    return r

X: (10962, 311)


## E23 — F&H 1 semente, base 311 (ref colega: 0.5008 na 339)

In [2]:
f1s, oof, _ = fh_oof(tag='e23_fh1s')
print(f'E23 cru: {f1s.mean():.4f} +/- {f1s.std():.4f}')
f1c = f1_score(y, corrigir(oof), average='macro')
print(f'E23 + correcao FAIXA_MINIMA (agregado): {f1c:.4f}')
registar('E23', 'F&H 1seed base-311', f1s, 0.5008, 'delta vs ref colega 339')

e23_fh1s: cache ../data/nb12_e23_fh1s.npz
E23 cru: 0.5022 +/- 0.0135
E23 + correcao FAIXA_MINIMA (agregado): 0.5032
{'exp': 'E23', 'variante': 'F&H 1seed base-311', 'f1': 0.502232970040124, 'std': 0.013519370933134134, 'delta': 0.001432970040123993, 'nota': 'delta vs ref colega 339'}


{'exp': 'E23',
 'variante': 'F&H 1seed base-311',
 'f1': 0.502232970040124,
 'std': 0.013519370933134134,
 'delta': 0.001432970040123993,
 'nota': 'delta vs ref colega 339'}

## E24 — F&H 1 semente, base 311 + quarentena-10 (rank NB11)

In [3]:
d = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))['proba']
rank = np.argsort(d[np.arange(len(y)), y])
drop10 = rank[:int(len(y) * 10 / 100)]
f1s, oof, _ = fh_oof(drop_idx=drop10, tag='e24_fh1s_q10')
print(f'E24 cru: {f1s.mean():.4f} +/- {f1s.std():.4f}')
f1c = f1_score(y, corrigir(oof), average='macro')
print(f'E24 + correcao (agregado): {f1c:.4f}')
registar('E24', 'F&H 1seed base-311 quar-10', f1s, 0.5008, 'delta vs ref colega 339')
np.savez_compressed(os.path.join(DATA_DIR, 'nb12_e24_f1s.npz'), f1s=f1s)


e24_fh1s_q10: cache ../data/nb12_e24_fh1s_q10.npz
E24 cru: 0.5168 +/- 0.0115
E24 + correcao (agregado): 0.5181
{'exp': 'E24', 'variante': 'F&H 1seed base-311 quar-10', 'f1': 0.5168268931520382, 'std': 0.011502054632972034, 'delta': 0.016026893152038202, 'nota': 'delta vs ref colega 339'}


## E25 — F&H 5 sementes + regressor HGB + correção, base 311 quar-10

Receita sub6 do colega (CV 0.5062 na 339): media das probas de 5 sementes (42,1,2,3,4), limiar 0.35 (max-proba < 0.35 vira regressor arredondado), correção max(pred, FAIXA_MINIMA). Variante mediana (conf < 0.40 vira mediana da distribuição) sem retreino.

In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor
SEEDS = [42, 1, 2, 3, 4]
d = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))['proba']
rank = np.argsort(d[np.arange(len(y)), y])
drop10 = set(rank[:int(len(y) * 10 / 100)].tolist())
p25 = os.path.join(DATA_DIR, 'nb12_e25_fh5.npz')
if os.path.exists(p25):
    dd = np.load(p25)
    Pmean, oof_seed = dd['Pmean'], dd['oof'].astype(int)
    print('e25 fh5: cache')
else:
    t0 = time.time()
    Pall = np.zeros((len(SEEDS), len(y), K))
    for si, sd in enumerate(SEEDS):
        _, _, pr = fh_oof(drop_idx=drop10, tag=f'e25_fh_s{sd}', seed=sd)
        Pall[si] = pr
        print(f'  seed {sd} done ({time.time()-t0:.0f}s)', flush=True)
    Pmean = Pall.mean(axis=0)
    oof_seed = Pmean.argmax(axis=1)
    np.savez_compressed(p25, Pmean=Pmean, oof=oof_seed)
f1s_fh5 = np.array([f1_score(y[va], oof_seed[va], average='macro') for _, va in FOLDS])
print(f'FH5 cru: {f1s_fh5.mean():.4f} +/- {f1s_fh5.std():.4f}')
e24mean = float(np.load(os.path.join(DATA_DIR, 'nb12_e24_f1s.npz'))['f1s'].mean())
registar('E25', 'FH 5seed quar-10 (cru)', f1s_fh5, e24mean, 'ref E24 1seed mesma base')
p25r = os.path.join(DATA_DIR, 'nb12_e25_reg.npz')
if os.path.exists(p25r):
    dd = np.load(p25r)
    oreg, rpred = dd['oof'].astype(int), dd['rval']
    print('e25 reg: cache')
else:
    t0 = time.time()
    oreg = np.zeros(len(y), dtype=int); rval = np.zeros(len(y))
    for fi, (tr, va) in enumerate(FOLDS):
        atr = np.array([i for i in tr if i not in drop10])
        r = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=15,
            min_samples_leaf=20, l2_regularization=1.0, early_stopping=True, random_state=RANDOM_STATE)
        r.fit(Xv[atr], y[atr], sample_weight=compute_sample_weight('balanced', y[atr]))
        rv = np.clip(np.round(r.predict(Xv[va])), 0, K - 1).astype(int)
        oreg[va] = rv; rval[va] = r.predict(Xv[va])
        print(f'  reg fold {fi} ({time.time()-t0:.0f}s)', flush=True)
    np.savez_compressed(p25r, oof=oreg, rval=rval)
pmax = Pmean.max(axis=1)
comb_reg = np.where(pmax < 0.35, oreg, oof_seed)
f1s_creg = np.array([f1_score(y[va], comb_reg[va], average='macro') for _, va in FOLDS])
print(f'FH5+reg@0.35: {f1s_creg.mean():.4f} +/- {f1s_creg.std():.4f}')
registar('E25', 'FH5+reg@0.35 quar-10', f1s_creg, float(f1s_fh5.mean()))
med = np.array([np.median(np.repeat(np.arange(K), (Pmean[i] * 1000).astype(int))) for i in range(len(y))]).astype(int)
comb_med = np.where(pmax < 0.40, np.clip(med, 0, K - 1), oof_seed)
f1s_cmed = np.array([f1_score(y[va], comb_med[va], average='macro') for _, va in FOLDS])
print(f'FH5+mediana@0.40: {f1s_cmed.mean():.4f} +/- {f1s_cmed.std():.4f}')
registar('E25', 'FH5+mediana@0.40 quar-10', f1s_cmed, float(f1s_fh5.mean()))
for nome, pred in [('FH5+reg@0.35+corr', corrigir(comb_reg)), ('FH5+med@0.40+corr', corrigir(comb_med))]:
    print(f'{nome} (agregado): {f1_score(y, pred, average="macro"):.4f}')


e25 fh5: cache
FH5 cru: 0.5195 +/- 0.0126
{'exp': 'E25', 'variante': 'FH 5seed quar-10 (cru)', 'f1': 0.5195148380116535, 'std': 0.012579820636043019, 'delta': 0.002687944859615321, 'nota': 'ref E24 1seed mesma base'}
e25 reg: cache
FH5+reg@0.35: 0.5209 +/- 0.0124
{'exp': 'E25', 'variante': 'FH5+reg@0.35 quar-10', 'f1': 0.520924600118281, 'std': 0.012434842064718987, 'delta': 0.0014097621066274657, 'nota': ''}


FH5+mediana@0.40: 0.5215 +/- 0.0117
{'exp': 'E25', 'variante': 'FH5+mediana@0.40 quar-10', 'f1': 0.5215133618518712, 'std': 0.011741859304256566, 'delta': 0.001998523840217703, 'nota': ''}
FH5+reg@0.35+corr (agregado): 0.5226
FH5+med@0.40+corr (agregado): 0.5230


## Submissão E25 (sub09): FH5 + mediana@0.40 + correção, treino total com quarentena

Fits finais no treino todo menos piores-10% (mesmo rank NB11). Mediana se max-proba < 0.40, senão argmax; correção max(pred, FAIXA_MINIMA do teste). Formato Kaggle: header AGE_RANGE, 3655 linhas, ordem do teste.

In [5]:
SUB_DIR = os.path.join(os.path.dirname(DATA_DIR), 'submissions')
te = pd.read_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
Xte = te.drop(columns=['PID']).to_numpy(dtype=np.float64)
fmin_te = te['FAIXA_MINIMA'].to_numpy().astype(int)
assert len(Xte) == 3655 and list(te.drop(columns=['PID']).columns) == list(train.drop(columns=['PID', TARGET]).columns)
d = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))['proba']
rank = np.argsort(d[np.arange(len(y)), y])
keep = np.setdiff1d(np.arange(len(y)), rank[:int(len(y) * 10 / 100)])
Xk, yk = Xv[keep], y[keep]
swk = compute_sample_weight('balanced', yk)
pte = os.path.join(DATA_DIR, 'nb12_e25_fh5_te.npz')
if os.path.exists(pte):
    Pte = np.load(pte)['Pmean_te']
    print('test probas: cache')
else:
    t0 = time.time()
    Pall = np.zeros((len(SEEDS), len(Xte), K))
    for si, sd in enumerate(SEEDS):
        ps = os.path.join(DATA_DIR, f'nb12_e25_fh5_te_s{sd}.npz')
        if os.path.exists(ps):
            Pall[si] = np.load(ps)['pf']
            print(f'  seed {sd} test: cache', flush=True)
            continue
        pg = np.zeros((len(Xte), K - 1))
        for k in range(K - 1):
            m = xgb_bin(sd)
            m.fit(Xk, (yk > k).astype(int), sample_weight=swk)
            pg[:, k] = m.predict_proba(Xte)[:, 1]
        pg = np.minimum.accumulate(pg, axis=1)
        pf = np.zeros((len(Xte), K))
        prev = np.ones(len(Xte))
        for k in range(K):
            cur = pg[:, k] if k < K - 1 else np.zeros(len(Xte))
            pf[:, k] = np.clip(prev - cur, 0, None)
            prev = cur
        Pall[si] = pf
        np.savez_compressed(ps, pf=pf)
        print(f'  seed {sd} test done ({time.time()-t0:.0f}s)', flush=True)
    Pte = Pall.mean(axis=0)
    np.savez_compressed(pte, Pmean_te=Pte)
from sklearn.ensemble import HistGradientBoostingRegressor
pre = os.path.join(DATA_DIR, 'nb12_e25_reg_te.npz')
if os.path.exists(pre):
    r_te = np.load(pre)['rval_te']
    print('reg test: cache')
else:
    r = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=20, l2_regularization=1.0, early_stopping=True, random_state=RANDOM_STATE)
    r.fit(Xk, yk, sample_weight=swk)
    r_te = r.predict(Xte)
    np.savez_compressed(pre, rval_te=r_te)
pred_arg = Pte.argmax(axis=1)
med_te = np.array([np.median(np.repeat(np.arange(K), (Pte[i] * 1000).astype(int))) for i in range(len(Xte))]).astype(int)
pred = np.where(Pte.max(axis=1) < 0.40, np.clip(med_te, 0, K - 1), pred_arg)
pred = np.maximum(pred, fmin_te).astype(int)
assert len(pred) == 3655 and pred.min() >= 0 and pred.max() <= 15
out = os.path.join(SUB_DIR, 'sub09_fh5_mediana_quar10.csv')
pd.DataFrame({'RowId': np.arange(len(pred)), 'AGE_RANGE': pred}).to_csv(out, index=False)
print('gravado:', out, pd.Series(pred).value_counts().sort_index().to_dict())


test probas: cache
reg test: cache


gravado: ../submissions/sub09_fh5_mediana_quar10.csv {0: 66, 1: 48, 2: 136, 3: 141, 4: 172, 5: 220, 6: 294, 7: 291, 8: 259, 9: 356, 10: 359, 11: 293, 12: 368, 13: 324, 14: 208, 15: 120}


## Vereditos (preencher após correr)

In [6]:
final = pd.DataFrame(RESULTADOS).sort_values('f1', ascending=False)
for c in ('f1', 'std', 'delta'):
    final[c] = final[c].round(4)
print(final.to_string(index=False))
final.to_csv(os.path.join(DATA_DIR, 'nb12_resultados.csv'), index=False)

exp                   variante     f1    std  delta                     nota
E25   FH5+mediana@0.40 quar-10 0.5215 0.0117 0.0020                         
E25       FH5+reg@0.35 quar-10 0.5209 0.0124 0.0014                         
E25     FH 5seed quar-10 (cru) 0.5195 0.0126 0.0027 ref E24 1seed mesma base
E24 F&H 1seed base-311 quar-10 0.5168 0.0115 0.0160  delta vs ref colega 339
E23         F&H 1seed base-311 0.5022 0.0135 0.0014  delta vs ref colega 339
